# Regression Diagnostics and Influential Observation Analysis for Real-World Prediction
**Statistical Machine Learning project · California Housing · Statsmodels OLS**

**Problem statement.** A regression model can score well on average and still be distorted by a few unusual observations. We fit a multiple linear regression, test its assumptions, detect outliers / high-leverage / influential observations, and measure how much they change the coefficients and predictions.

**Workflow.** Load → split (train/test) → EDA → OLS on **train only** → assumption diagnostics → influence analysis on **train only** → sensitivity model → compare both on the **same untouched test set** → save results.

> All calculations live in `sml_core.py` (shared with the Streamlit dashboard), so the notebook and the dashboard always agree.
> Nothing below is hard-coded: every number and interpretation is produced by running the code.

## 0. Setup (works on Google Colab and local Windows)

In [ ]:
import os, sys, subprocess

# ---- Google Colab helper: install missing packages and make sure sml_core.py is available ----
IN_COLAB = "google.colab" in sys.modules
try:
    import statsmodels  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels", "scipy", "seaborn", "scikit-learn"], check=True)

if not os.path.exists("sml_core.py"):
    if IN_COLAB:
        from google.colab import files
        print("Please upload sml_core.py (from the project folder) ...")
        files.upload()
    else:
        raise FileNotFoundError("Open this notebook from inside the Regression_Diagnostics_Project folder.")

%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sml_core as core

pd.set_option("display.max_columns", 50); pd.set_option("display.precision", 4)
np.random.seed(core.RANDOM_STATE)
core.ensure_dirs()
print("Setup OK | random seed =", core.RANDOM_STATE)

## Module 1 · Data collection and preprocessing
**Dataset:** California Housing (1990 census, one row per block group), loaded with `sklearn.datasets.fetch_california_housing` - no manual download.

**Target `MedHouseVal`** = median house value in units of **$100,000** (2.5 = $250,000). It is **capped (censored) at 5.0**, so every block group worth more than $500,000 appears as exactly 5.0.

In [ ]:
df = core.load_data()
ov = core.data_overview(df)
print("Dimensions (rows, columns):", ov["shape"])
print("Duplicate rows:", ov["n_duplicates"])
print(f"Rows at the target cap ({core.TARGET_CAP}): {ov['n_at_cap']:,} ({ov['pct_at_cap']:.2f}%)")
display(ov["dtypes"].T)
display(ov["missing"].T)
display(ov["describe"])
display(df.head())

In [ ]:
# Reproducible split. The TEST set is locked away: it is never used for fitting,
# influence analysis, or deciding which observations to flag (no data leakage).
X_train, X_test, y_train, y_test = core.split_data(df)
print(f"Train: {X_train.shape}   Test: {X_test.shape}   (test size {core.TEST_SIZE:.0%}, seed {core.RANDOM_STATE})")

## Module 2 · Exploratory Data Analysis

In [ ]:
core.plot_target_dist(df, f"{core.DIAG_DIR}/01_target_distribution.png"); plt.show()
core.plot_feature_dists(df, f"{core.DIAG_DIR}/02_feature_distributions.png"); plt.show()

In [ ]:
core.plot_corr(df, f"{core.DIAG_DIR}/03_correlation_heatmap.png"); plt.show()
core.plot_feature_vs_target(df, f"{core.DIAG_DIR}/04_feature_vs_target.png"); plt.show()

In [ ]:
print("Statistical observations (computed from the data):")
for o in core.eda_observations(df): print(" -", o)

## Module 3 · Multiple linear regression (Statsmodels OLS)
Model: $y = \beta_0 + \beta_1 x_1 + \dots + \beta_8 x_8 + \varepsilon$, estimated by minimising $\sum e_i^2$.

* **Coefficient** $\beta_j$: expected change in house value (×$100,000) for +1 unit of feature $j$, others fixed.
* **p-value**: probability of a coefficient this extreme if the true $\beta_j = 0$.
* **R²** = share of variance explained; **adjusted R²** penalises extra features.

In [ ]:
orig = core.fit_ols(X_train, y_train)          # fitted on TRAINING data only
print(orig.summary())

In [ ]:
display(core.coefficient_table(orig))
test_pred = core.predict(orig, X_test)
metrics = {"train": core.regression_metrics(y_train, np.asarray(orig.fittedvalues)),
           "test":  core.regression_metrics(y_test, test_pred)}
display(pd.DataFrame(metrics).T.rename_axis("split"))
display(Markdown("**Interpretation (from the actual output):** " + core.interpret_model(orig, metrics["test"])))

In [ ]:
core.plot_actual_vs_pred(y_test, test_pred, "Actual vs predicted - TEST set", f"{core.DIAG_DIR}/05_actual_vs_predicted_test.png"); plt.show()

## Module 4 · Regression diagnostics
OLS inference (t-tests, p-values, confidence intervals) relies on assumptions about the errors: **linearity, constant variance, normality, no strong multicollinearity**. We check each one on the training residuals.

### 4.1 Residuals vs fitted values
* **Checks:** linearity and constant variance. **Why it matters:** non-linearity means a mis-specified model; a funnel means unreliable standard errors.
* **Read it:** a flat random cloud around 0 is good. Curvature = non-linear pattern; a widening funnel = heteroscedasticity.

In [ ]:
core.plot_resid_vs_fitted(orig.fittedvalues, orig.resid, f"{core.DIAG_DIR}/06_residuals_vs_fitted.png"); plt.show()

### 4.2 Residual histogram and 4.3 Normal Q-Q plot
* **Check:** whether residuals are approximately normal. **Why:** needed for exact t-tests and prediction intervals (less critical for large samples, but heavy tails mean some predictions are far off).
* **Read:** histogram close to the red curve; Q-Q points on the straight line. Curved ends = heavy tails / skew.

In [ ]:
core.plot_resid_hist(orig.resid, f"{core.DIAG_DIR}/07_residual_histogram.png"); plt.show()
core.plot_qq(orig.resid, f"{core.DIAG_DIR}/08_qq_plot.png"); plt.show()

### 4.4 Breusch-Pagan and 4.5 Jarque-Bera tests, 4.6 VIF
* **Breusch-Pagan** - H0: constant error variance. Regress $e_i^2$ on the predictors; $LM = nR^2_{aux} \sim \chi^2_k$. Small p ⇒ heteroscedasticity.
* **Jarque-Bera** - H0: residuals are normal. $JB = \frac{n}{6}\left(S^2 + \frac{(K-3)^2}{4}\right) \sim \chi^2_2$. Small p ⇒ non-normal.
* **VIF** - $VIF_j = \frac{1}{1-R_j^2}$ where $R_j^2$ comes from regressing feature $j$ on the other features. >5 moderate, >10 serious multicollinearity (unstable individual coefficients).

In [ ]:
diag = core.run_diagnostics(orig, X_train)
display(pd.DataFrame({"Breusch-Pagan": diag["bp"]}).T)
display(pd.DataFrame({"Jarque-Bera": diag["jb"]}).T)
display(diag["vif"])
core.plot_vif(diag["vif"], f"{core.DIAG_DIR}/09_vif.png"); plt.show()
for key, txt in [("Breusch-Pagan", core.interpret_bp(diag)), ("Jarque-Bera", core.interpret_jb(diag)), ("VIF", core.interpret_vif(diag["vif"]))]:
    display(Markdown(f"**{key} - what the actual result means:** {txt}"))

## Module 5 · Influential observation analysis
| Concept | Meaning | Measured by |
|---|---|---|
| **Outlier** | unusual **y** given its x (large residual) | externally studentized residual, \|t\| > 3 |
| **High-leverage point** | unusual **x** (far from the centre of the predictors) | hat value $h_{ii}$ > 2p/n |
| **Influential observation** | removing it noticeably changes the fit | Cook's D > 4/n, \|DFFITS\| > 2√(p/n) |

A point can be high-leverage but harmless (it lies on the trend) or an outlier with low leverage (little pull). **Influence ≈ outlyingness × leverage.**

$$D_i = \frac{e_i^2}{p\,s^2}\cdot\frac{h_{ii}}{(1-h_{ii})^2}\qquad DFFITS_i = t_i\sqrt{\frac{h_{ii}}{1-h_{ii}}}\qquad t_i=\frac{e_i}{s_{(i)}\sqrt{1-h_{ii}}}$$

**Important:** crossing a threshold only makes an observation *potentially* influential. We investigate before deciding anything - nothing is deleted automatically. With ~16,000 rows the 4/n rule flags many points, which is why we also show stricter rules in Module 6.

In [ ]:
infl = core.influence_table(orig, X_train, y_train)     # TRAINING data only
th = core.influence_thresholds(orig)
infl.attrs["thresholds"] = th
print("Thresholds:", {k: round(v, 6) if isinstance(v, float) else v for k, v in th.items()})
flag_cols = ["Flag_Cook", "Flag_Leverage", "Flag_Outlier", "Flag_DFFITS"]
display(infl[flag_cols].sum().rename("count flagged").to_frame().T)
display(infl["Type"].value_counts().to_frame("count"))

In [ ]:
core.plot_cooks(infl, th, f"{core.INFL_DIR}/cooks_distance.png"); plt.show()
core.plot_leverage_resid(infl, th, f"{core.INFL_DIR}/leverage_vs_studentized_residual.png"); plt.show()

In [ ]:
print("Top 15 observations ranked by Cook's distance (Obs_ID = row id in the original dataset)")
display(infl[["Cook_Rank","Obs_ID","Cook_D","Leverage","Studentized_Resid","DFFITS","Actual","Fitted","Type"] + core.FEATURES].head(15))

In [ ]:
RULE = "cook"      # flagging rule: "cook" (default), "dffits" or "both"
flagged = core.flag_observations(infl, RULE)
display(Markdown("**Influence summary:** " + core.interpret_influence(infl, th, flagged, RULE)))

# INVESTIGATE before acting: what do the flagged observations look like?
profile, type_counts = core.investigate_flagged(infl, flagged)
display(type_counts)
display(profile)

**Reading the investigation table.** Compare the flagged and not-flagged columns. Flagged blocks usually have unusual feature values (e.g. extreme occupancy or room counts) or sit at the price cap. They are *real* data, not errors, so the right question is *"how much do they matter?"* - which we answer next by refitting without them (a sensitivity analysis), not by assuming they should be removed.

## Module 6 · Sensitivity model and comparison
1. Original model: all training data.  2. Sensitivity model: training data **minus** flagged observations.
3. **Both** are scored on the **exact same untouched test set**, so the comparison is fair and leak-free.

In [ ]:
sens, keep_mask = core.fit_sensitivity(X_train, y_train, infl, flagged)
print(f"Training rows: original = {len(X_train):,} | sensitivity = {int(keep_mask.sum()):,} | removed = {int((~keep_mask).sum()):,}")
comp = core.compare_models(orig, sens, X_test, y_test, X_train, y_train, keep_mask)
print("TEST-set metrics (same test set for both models):")
display(comp["test_metrics"])
core.plot_metrics_compare(comp["test_metrics"], f"{core.COMP_DIR}/test_metrics_comparison.png"); plt.show()

In [ ]:
display(comp["coef"])
core.plot_coef_compare(comp["coef"], f"{core.COMP_DIR}/coefficient_changes.png"); plt.show()
display(Markdown("**Verdict (from the actual numbers):** " + core.interpret_comparison(comp, int(flagged.sum()))))

In [ ]:
# Does the conclusion depend on how we flag? (test metrics are REPORTED only - never used to pick a rule)
sweep = core.threshold_sweep(X_train, y_train, X_test, y_test, infl)
display(sweep)

## Module 7 · Interactive Streamlit dashboard
The dashboard uses the same `sml_core.py` functions. Run it from a terminal in the project folder:

```
streamlit run app.py
```
(See `README.md` for Windows, Colab and Streamlit Community Cloud instructions.)

## Module 8 · Save results, consistency check, discussion

In [ ]:
# Consistency check: the step-by-step notebook must equal the one-call pipeline used by the dashboard.
res = core.run_analysis(rule=RULE)
assert np.isclose(res["comp"]["test_metrics"].loc["RMSE", "Original"], comp["test_metrics"].loc["RMSE", "Original"])
assert np.isclose(res["comp"]["test_metrics"].loc["RMSE", "Sensitivity"], comp["test_metrics"].loc["RMSE", "Sensitivity"])
assert int(res["flagged"].sum()) == int(flagged.sum())
print("Notebook and dashboard pipeline agree.")

files = core.save_all_outputs(res)
with open(os.path.join(core.REPORT_DIR, "project_results.md"), "w", encoding="utf-8") as f:
    f.write(core.build_results_markdown(res))
print(f"Saved {len(files)} files to outputs/ and reports/project_results.md")

In [ ]:
summary = pd.DataFrame({
    "Original": comp["test_metrics"]["Original"], "Sensitivity": comp["test_metrics"]["Sensitivity"],
    "Change": comp["test_metrics"]["Change"], "% change": comp["test_metrics"]["Pct_Change"]})
display(Markdown("### Results comparison table (test set)"))
display(summary)
display(Markdown("### Conclusion\n" + core.interpret_comparison(comp, int(flagged.sum()))))

### Limitations
* Linear model: cannot represent the non-linear location effect or interactions.
* The target is censored at 5.0; OLS treats capped values as real values.
* Block-group averages from 1990 - not individual houses and not current prices.
* Influence thresholds (4/n, 2p/n, |t|>3, 2√(p/n)) are conventions; with large n they flag many points.
* Masking: groups of similar extreme points can hide each other in one-at-a-time diagnostics.

### Future enhancements
Robust regression (Huber/RANSAC), HC3 robust standard errors, log-transform of the target, spatial or tree-based models, DFBETAS per coefficient, a Tobit model for the censored target, and cross-validated influence analysis.